In [4]:
"""

This module handles dataset preprocessing, conversion into binary/multi-class formats,
and computation of adjusted evaluation metrics for outlier detection algorithms.

Key Features:
  - IDF transformation for categorical attributes
  - Data preprocessing (missing values, duplicates, normalization)
  - Multiple conversion methods: BIN, BINDOWN, EXC, EXCDOWN, GRO, GRODOWN
  - Calculation of adjusted ranking-based metrics (AUC, P@n, AP, Max-F1)
  - Statistical aggregation and LaTeX export

Organization:
  1. Imports & Dependencies
  2. Configuration & Constants
  3. Data Loading & Preprocessing
  4. Dataset Conversion Methods
  5. Main Execution Pipeline
"""

import os
import re
import math
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
import warnings
from tqdm import tqdm

# Suppress all library warnings for cleaner output
warnings.filterwarnings("ignore")

In [7]:
# =============================================================================
# SECTION 1: Configuration & Constants
# =============================================================================

# Default outlier rate for downsampling operations (percentage)
DEFAULT_OUTLIER_RATE = 5

# Number of versions to generate for downsampling methods
NUM_VERSIONS = 5

# Conversion method categories
CONVERSION_METHODS = ['BIN', 'BINDOWN', 'EXC', 'EXCDOWN', 'GRO', 'GRODOWN']


# =============================================================================
# SECTION 2: Data Loading & Preprocessing
# =============================================================================

def apply_idf_transformation(data: pd.DataFrame, attribute: str) -> pd.DataFrame:
    """
    Apply Inverse Document Frequency (IDF) transformation to categorical attribute.
    
    Converts categorical values to their IDF weights: ln(N/frequency)
    This gives higher weights to rare values and lower weights to common ones.
    
    Args:
        data: DataFrame containing the attribute to transform
        attribute: Column name to apply IDF transformation
        
    Returns:
        DataFrame with IDF-transformed attribute
    """
    N = len(data)
    
    # Calculate frequency of each unique value
    frequency_table = data[attribute].value_counts()
    
    if N == 0:
        return data
    
    # Calculate ln(N/frequency) for each value
    idf_weights = {}
    for value, frequency in frequency_table.items():
        idf_weights[value] = math.log(N / frequency)
    
    # Replace categorical values with their IDF weights
    for key in idf_weights.keys():
        data.loc[data[attribute] == key, attribute] = idf_weights[key]
    
    return data


def preprocess_dataset(data: pd.DataFrame) -> pd.DataFrame:
    """
    Perform comprehensive data preprocessing.
    
    Steps:
    1. Remove rows with missing values (NaN or '?')
    2. Remove duplicate rows
    3. Apply IDF transformation to categorical attributes
    4. Normalize numeric features to [0, 1]
    5. Rename 'class' column to 'outlier' if present
    
    Args:
        data: Raw DataFrame to preprocess
        
    Returns:
        Cleaned and normalized DataFrame
    """
    # Step 1: Remove rows with missing values
    data = data.dropna()
    data = data[(data != '?').all(axis=1)]
    
    # Step 2: Remove duplicate rows
    data = data.drop_duplicates().reset_index(drop=True)
    
    # Step 3: Apply IDF transformation to categorical attributes
    for column in data.columns[:-1]:
        if data[column].dtypes == 'object' or data[column].dtypes == 'categorical':
            data = apply_idf_transformation(data, column)
    
    # Step 4: Normalize features to [0, 1] range (excluding class label)
    normalized_features = pd.DataFrame(
        MinMaxScaler().fit_transform(data.iloc[:, :-1]),
        columns=data.columns[:-1]
    )
    
    # Step 5: Rename 'class' to 'outlier' and attach back to normalized data
    if 'class' in data.columns:
        data.rename(columns={'class': 'outlier'}, inplace=True)
    
    normalized_features['outlier'] = data['outlier']
    
    return normalized_features


# =============================================================================
# SECTION 3: Dataset Conversion Methods
# =============================================================================

def get_class_distribution(df: pd.DataFrame) -> tuple:
    """
    Extract class distribution information from dataset.
    
    Identifies majority and minority classes and their counts.
    
    Args:
        df: DataFrame with 'outlier' column
        
    Returns:
        Tuple of (class_inlier, count_inlier, class_outlier, count_outlier, total_instances)
    """
    # Count frequency of each class (sorted descending)
    class_column = df.columns[-1]
    class_counts = df[class_column].value_counts(ascending=False)
    
    class_inlier = class_counts.index[0]      # Most frequent (majority/inlier)
    count_inlier = class_counts.values[0]
    class_outlier = class_counts.index[-1]    # Least frequent (minority/outlier)
    count_outlier = class_counts.values[-1]
    total_instances = len(df)
    
    return class_inlier, count_inlier, class_outlier, count_outlier, total_instances


def convert_binary_labels(df: pd.DataFrame, class_outlier) -> pd.DataFrame:
    """
    Convert any class labels to binary 'yes'/'no' format.
    
    Args:
        df: DataFrame with 'outlier' column
        class_outlier: Original label representing outlier class
        
    Returns:
        DataFrame with binary outlier labels
    """
    class_column = df.columns[-1]
    df[class_column] = df[class_column].apply(
        lambda x: 'yes' if x == class_outlier else 'no'
    )
    return df


def save_converted_dataset(df: pd.DataFrame, output_path: str) -> None:
    """
    Save converted dataset to CSV file with semicolon separator.
    
    Args:
        df: DataFrame to save
        output_path: Full path including filename
    """
    if 'class' in df.columns:
        df.rename(columns={'class': 'outlier'}, inplace=True)
    df.to_csv(output_path, sep=';', index=False)


def apply_downsampling(df: pd.DataFrame, class_inlier: int, count_inlier: int,
                      class_outlier: int, outlier_rate: float) -> pd.DataFrame:
    """
    Apply downsampling to balance dataset.
    
    Randomly removes outlier instances until target ratio is reached.
    
    Args:
        df: DataFrame to downsample
        class_inlier: Label of majority class
        count_inlier: Count of inlier instances
        class_outlier: Label of minority class
        outlier_rate: Target outlier percentage relative to inliers
        
    Returns:
        Downsampled DataFrame
    """
    # Calculate target total size based on inlier count and outlier rate
    target_size = count_inlier + (count_inlier * outlier_rate / 100)
    
    # Randomly remove outliers until target size reached
    class_column = df.columns[-1]
    while len(df) > target_size:
        outliers = df[df[class_column] == class_outlier]
        
        if len(outliers) == 0:
            break
        
        # Select and remove a random outlier instance
        row_to_remove = outliers.sample(n=1).index
        df = df.drop(index=row_to_remove)
    
    return df


def convert_bin(df: pd.DataFrame, dataset_name: str, output_base_path: str) -> None:
    """
    BIN Conversion: Convert labels to binary format without modification.
    
    Simple conversion of any multi-class dataset to binary 'yes'/'no' labels.
    
    Args:
        df: Preprocessed DataFrame
        dataset_name: Name of dataset file
        output_base_path: Base directory path for output
    """
    class_inlier, _, class_outlier, _, _ = get_class_distribution(df)
    df = convert_binary_labels(df, class_outlier)
    
    os.makedirs(os.path.join(output_base_path, 'BIN'), exist_ok=True)
    output_path = os.path.join(output_base_path, 'BIN', dataset_name)
    save_converted_dataset(df, output_path)


def convert_bindown(df: pd.DataFrame, dataset_name: str, output_base_path: str,
                   outlier_rate: float = DEFAULT_OUTLIER_RATE,
                   num_versions: int = NUM_VERSIONS) -> None:
    """
    BINDOWN Conversion: Binary conversion with downsampling.
    
    Converts to binary and creates multiple versions with progressive downsampling.
    
    Args:
        df: Preprocessed DataFrame
        dataset_name: Name of dataset file
        output_base_path: Base directory path for output
        outlier_rate: Target outlier percentage for downsampling
        num_versions: Number of downsampled versions to create
    """
    class_inlier, count_inlier, class_outlier, _, _ = get_class_distribution(df)
    
    # Generate multiple downsampled versions
    for version in range(num_versions):
        df_copy = df.copy()
        
        # Apply downsampling
        df_copy = apply_downsampling(df_copy, class_inlier, count_inlier,
                                     class_outlier, outlier_rate)
        
        # Convert to binary labels
        df_copy = convert_binary_labels(df_copy, class_outlier)
        
        # Save with version suffix
        output_filename = dataset_name.replace('.csv', f'_v{version+1}.csv')
        os.makedirs(os.path.join(output_base_path, 'BINDOWN'), exist_ok=True)
        output_path = os.path.join(output_base_path, 'BINDOWN', output_filename)
        save_converted_dataset(df_copy, output_path)


def convert_exclude(df: pd.DataFrame, dataset_name: str, output_base_path: str) -> None:
    """
    EXC Conversion: Exclude intermediate classes, keep only majority vs minority.
    
    Removes instances from intermediate classes for multi-class datasets.
    
    Args:
        df: Preprocessed DataFrame
        dataset_name: Name of dataset file
        output_base_path: Base directory path for output
    """
    class_inlier, _, class_outlier, _, _ = get_class_distribution(df)
    
    # Remove intermediate classes
    excluded_mask = (df['outlier'] != class_inlier) & (df['outlier'] != class_outlier)
    rows_to_remove = df[excluded_mask].index
    df = df.drop(index=rows_to_remove)
    
    # Convert to binary labels
    df = convert_binary_labels(df, class_outlier)
    
    os.makedirs(os.path.join(output_base_path, 'EXC'), exist_ok=True)
    output_path = os.path.join(output_base_path, 'EXC', dataset_name)
    save_converted_dataset(df, output_path)


def convert_exclude_down(df: pd.DataFrame, dataset_name: str, output_base_path: str,
                        outlier_rate: float = DEFAULT_OUTLIER_RATE,
                        num_versions: int = NUM_VERSIONS) -> None:
    """
    EXCDOWN Conversion: Exclude intermediate classes + downsampling.
    
    Removes intermediate classes and creates multiple downsampled versions.
    
    Args:
        df: Preprocessed DataFrame
        dataset_name: Name of dataset file
        output_base_path: Base directory path for output
        outlier_rate: Target outlier percentage for downsampling
        num_versions: Number of downsampled versions to create
    """
    class_inlier, count_inlier, class_outlier, _, _ = get_class_distribution(df)
    
    # Remove intermediate classes
    excluded_mask = (df['outlier'] != class_inlier) & (df['outlier'] != class_outlier)
    rows_to_remove = df[excluded_mask].index
    df = df.drop(index=rows_to_remove)
    
    # Generate multiple downsampled versions
    for version in range(num_versions):
        df_copy = df.copy()
        
        # Apply downsampling
        df_copy = apply_downsampling(df_copy, class_inlier, count_inlier,
                                     class_outlier, outlier_rate)
        
        # Convert to binary labels
        df_copy = convert_binary_labels(df_copy, class_outlier)
        
        # Save with version suffix
        output_filename = dataset_name.replace('.csv', f'_v{version+1}.csv')
        os.makedirs(os.path.join(output_base_path, 'EXCDOWN'), exist_ok=True)
        output_path = os.path.join(output_base_path, 'EXCDOWN', output_filename)
        save_converted_dataset(df_copy, output_path)


def convert_grouping(df: pd.DataFrame, dataset_name: str, output_base_path: str) -> None:
    """
    GRO Conversion: Group intermediate classes with majority class.
    
    Merges all non-outlier classes together for binary distinction.
    
    Args:
        df: Preprocessed DataFrame
        dataset_name: Name of dataset file
        output_base_path: Base directory path for output
    """
    class_inlier, _, class_outlier, _, _ = get_class_distribution(df)
    
    # Group intermediate classes with majority class
    df['outlier'] = df['outlier'].apply(
        lambda x: class_inlier if x != class_outlier else x
    )
    
    # Convert to binary labels
    df = convert_binary_labels(df, class_outlier)
    
    os.makedirs(os.path.join(output_base_path, 'GRO'), exist_ok=True)
    output_path = os.path.join(output_base_path, 'GRO', dataset_name)
    save_converted_dataset(df, output_path)


def convert_grouping_down(df: pd.DataFrame, dataset_name: str, output_base_path: str,
                         outlier_rate: float = DEFAULT_OUTLIER_RATE,
                         num_versions: int = NUM_VERSIONS) -> None:
    """
    GRODOWN Conversion: Group intermediate classes + downsampling.
    
    Groups intermediate classes with majority and creates multiple downsampled versions.
    
    Args:
        df: Preprocessed DataFrame
        dataset_name: Name of dataset file
        output_base_path: Base directory path for output
        outlier_rate: Target outlier percentage for downsampling
        num_versions: Number of downsampled versions to create
    """
    class_inlier, count_inlier, class_outlier, _, _ = get_class_distribution(df)
    
    # Group intermediate classes with majority class
    df['outlier'] = df['outlier'].apply(
        lambda x: class_inlier if x != class_outlier else x
    )
    
    # Generate multiple downsampled versions
    for version in range(num_versions):
        df_copy = df.copy()
        
        # Apply downsampling
        df_copy = apply_downsampling(df_copy, class_inlier, count_inlier,
                                     class_outlier, outlier_rate)
        
        # Convert to binary labels
        df_copy = convert_binary_labels(df_copy, class_outlier)
        
        # Save with version suffix
        output_filename = dataset_name.replace('.csv', f'_v{version+1}.csv')
        os.makedirs(os.path.join(output_base_path, 'GRODOWN'), exist_ok=True)
        output_path = os.path.join(output_base_path, 'GRODOWN', output_filename)
        save_converted_dataset(df_copy, output_path)

def convert_all_binaries(df: pd.DataFrame, dataset_name: str, output_base_path: str) -> None:
    """
    Apply all conversion methods to the binary datasets.
    
    Args:
        df: Preprocessed DataFrame
        dataset_name: Name of dataset file
        output_base_path: Base directory path for output
    """
    if 'class' in df.columns:
        df.rename(columns={'class': 'outlier'}, inplace=True)
    convert_bin(df, dataset_name, output_base_path)
    convert_bindown(df, dataset_name, output_base_path)
    
def convert_all_non_binaries(df: pd.DataFrame, dataset_name: str, output_base_path: str) -> None:
    """
    Apply all conversion methods to the non_binary datasets.
    
    Args:
        df: Preprocessed DataFrame
        dataset_name: Name of dataset file
        output_base_path: Base directory path for output
    """
    if 'class' in df.columns:
        df.rename(columns={'class': 'outlier'}, inplace=True)
    convert_exclude(df, dataset_name, output_base_path)
    convert_exclude_down(df, dataset_name, output_base_path)
    convert_grouping(df, dataset_name, output_base_path)
    convert_grouping_down(df, dataset_name, output_base_path)

In [8]:
# =============================================================================
# SECTION 4: Main Execution Pipeline
# =============================================================================

def main():
    """
    Main execution pipeline.
    
    Orchestrates complete workflow:
    1. Dataset preprocessing and conversion
    2. Metrics calculation
    3. Statistical analysis and export
    """
    print("\n" + "="*70)
    print("Dataset Conversion and Metrics Calculation Pipeline")
    print("="*70)
    
    # Configuration
    base_dir = 'conversion_methods'
    datasets_dir = os.path.join(r'..\..\datasets', base_dir)
    os.makedirs(os.sep.join([r'..\..\results', base_dir]), exist_ok=True)
    results_dir = os.path.join(r'..\..\results', base_dir)
    
    # Conversion functions mapping
    # Binary
    print('Convert Binary Datasets')
    for dataset in os.listdir(os.sep.join([datasets_dir, 'binary'])):
        file_path = os.sep.join([datasets_dir, 'binary', dataset])
        if os.path.isfile(file_path) and dataset.endswith('.csv'):
            df = pd.read_csv(file_path, sep=';')
            df = preprocess_dataset(df)
            convert_all_binaries(df, dataset, os.sep.join([datasets_dir, 'binary', 'converted']))
    # Non_Binary
    print('Convert Non-Binary Datasets')
    for dataset in os.listdir(os.sep.join([datasets_dir, 'non_binary'])):
        file_path = os.sep.join([datasets_dir, 'non_binary', dataset])
        if os.path.isfile(file_path) and dataset.endswith('.csv'):
            df = pd.read_csv(file_path, sep=';')
            df = preprocess_dataset(df)
            convert_all_non_binaries(df, dataset, os.sep.join([datasets_dir, 'non_binary', 'converted']))
    
    print("\n" + "="*70)
    print("Pipeline Complete")
    print("="*70 + "\n")


if __name__ == "__main__":
    main()


Dataset Conversion and Metrics Calculation Pipeline
Convert Binary Datasets
Convert Non-Binary Datasets

Pipeline Complete

